In [1]:
# ==============================================================================
# ENTERPRISE P&L PIPELINE: SYNERION & UKG TIME AND COSTS ENRICHMENT
# ==============================================================================
import requests
import time
import uuid
import traceback
from datetime import datetime, timedelta, date
from typing import List, Dict, Any, Optional
from zoneinfo import ZoneInfo

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, max as _max, lit, lower, trim, from_utc_timestamp, substring, 
    coalesce, when, to_date, sum, split, explode, sequence, 
    expr, trunc, concat, first, row_number, desc, current_timestamp, lead
)
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ------------------------------------------------------------------------------
# 1. PIPELINE CONFIGURATION
# ------------------------------------------------------------------------------
class PipelineConfig:
    SCHEMA_AUDIT    = "audit"
    PIPELINE_NAME   = "NB_Corp_Finance_DataHub_ELT_Synerion_UKG" 
    ENVIRONMENT     = "Production"
    TZ              = "America/New_York"
    RUN_ID          = str(uuid.uuid4())
    TARGET_CATALOG  = "LH_Corp_DataHub"
    
    # Schema Definitions
    SCHEMA_BRONZE_SYN = "synerion"
    SCHEMA_BRONZE_HR  = "hr"
    SCHEMA_SILVER     = "silver"

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
RUN_FULL_LOAD = True 

# ------------------------------------------------------------------------------
# 2. TELEMETRY & TIME UTILITIES
# ------------------------------------------------------------------------------
def _est_naive_now() -> datetime:
    return datetime.now(ZoneInfo(PipelineConfig.TZ)).replace(tzinfo=None)

def _est_now_str() -> str:
    return datetime.now(ZoneInfo(PipelineConfig.TZ)).strftime("%Y-%m-%d %H:%M:%S")

def log_info(msg):    print(f"{_est_now_str()} - [INFO] - {msg}")
def log_error(msg):   print(f"{_est_now_str()} - [ERROR] - {msg}")
def log_success(msg): print(f"{_est_now_str()} - [SUCCESS] - {msg}")

# ------------------------------------------------------------------------------
# 3. GOVERNANCE & AUDIT MANAGER
# ------------------------------------------------------------------------------
class AuditManager:
    """
    Maintains a verifiable chain of custody for all financial data transformations.
    Writes telemetry to Delta Lake in real-time for observability.
    """
    LOG_SCHEMA = StructType([
        StructField("RunID",              StringType(),    False),
        StructField("PipelineName",       StringType(),    False),
        StructField("Environment",        StringType(),    False),
        StructField("StartTime",          TimestampType(), False),
        StructField("EndTime",            TimestampType(), True),
        StructField("OverallStatus",      StringType(),    False),
        StructField("TotalRowsProcessed", LongType(),      True),
        StructField("ErrorMessage",       StringType(),    True),
    ])

    DETAIL_SCHEMA = StructType([
        StructField("DetailID",           StringType(),    False),
        StructField("RunID",              StringType(),    False),
        StructField("EntityName",         StringType(),    False),
        StructField("Layer",              StringType(),    True),
        StructField("Operation",          StringType(),    True),
        StructField("RowCount",           LongType(),      True),
        StructField("Status",             StringType(),    True),
        StructField("Message",            StringType(),    True),
        StructField("LogTime",            TimestampType(), True),
        StructField("Timestamp",          TimestampType(), True),
        StructField("Environment",        StringType(),    True),
        StructField("StartTime",          TimestampType(), True),
        StructField("EndTime",            TimestampType(), True),
        StructField("RowsInserted",       LongType(),      True),
        StructField("RowsUpdated",        LongType(),      True),
        StructField("DurationSeconds",    LongType(),      True),
    ])

    def __init__(self, *, spark: SparkSession, config: type):
        self._spark   = spark
        self._cfg     = config
        self._details: List[Dict[str, Any]] = []
        self._total   = 0
        self._t0      = _est_naive_now()
        self._seq     = 0
        self._spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PipelineConfig.TARGET_CATALOG}.{self._cfg.SCHEMA_AUDIT}")

    def log_run_start(self) -> None:
        log_tbl = f"{self._cfg.TARGET_CATALOG}.{self._cfg.SCHEMA_AUDIT}.run_audit_log"
        data = [{
            "RunID": self._cfg.RUN_ID, "PipelineName": self._cfg.PIPELINE_NAME,
            "Environment": self._cfg.ENVIRONMENT, "StartTime": self._t0,
            "EndTime": None, "OverallStatus": "RUNNING",
            "TotalRowsProcessed": 0, "ErrorMessage": None,
        }]
        try:
            self._spark.createDataFrame(data, self.LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(log_tbl)
        except Exception as e:
            log_error(f"Audit tracking initialization failed: {str(e)[:300]}")

    def log_run_end(self, *, status: str, error_message: str = "") -> None:
        log_tbl = f"{self._cfg.TARGET_CATALOG}.{self._cfg.SCHEMA_AUDIT}.run_audit_log"
        end = _est_naive_now()
        try:
            self._spark.sql(f"REFRESH TABLE {log_tbl}")
            dt = DeltaTable.forName(self._spark, log_tbl)
            dt.update(
                condition=f"RunID = '{self._cfg.RUN_ID}'",
                set={
                    "EndTime": lit(end), "OverallStatus": lit(status),
                    "TotalRowsProcessed": lit(self._total),
                    "ErrorMessage": lit(error_message[:4000]) if error_message else lit(None),
                }
            )
        except Exception as e:
            log_error(f"Audit ledger finalization failed: {str(e)[:300]}")

    def log_detail(
        self, *, entity_name: str, layer: str, operation: str, row_count: int = 0, 
        status: str = "OK", message: str = "", rows_inserted: int = 0, 
        rows_updated: int = 0, start_time: Optional[datetime] = None,
    ) -> None:
        self._seq += 1
        now = _est_naive_now()
        self._details.append({
            "DetailID": f"{self._cfg.RUN_ID}_{self._seq:04d}", "RunID": self._cfg.RUN_ID,
            "EntityName": entity_name, "Layer": layer, "Operation": operation,
            "RowCount": row_count, "Status": status, "Message": message[:4000],
            "LogTime": now, "Timestamp": now, "Environment": self._cfg.ENVIRONMENT,
            "StartTime": start_time if start_time else now, "EndTime": now,
            "RowsInserted": rows_inserted, "RowsUpdated": rows_updated,
            "DurationSeconds": int((now - start_time).total_seconds()) if start_time else 0,
        })
        if status == "OK": self._total += row_count
        self.flush_details()

    def flush_details(self) -> None:
        if not self._details: return
        detail_tbl = f"{self._cfg.TARGET_CATALOG}.{self._cfg.SCHEMA_AUDIT}.run_audit_detail"
        try:
            (self._spark.createDataFrame(self._details, self.DETAIL_SCHEMA)
                .write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(detail_tbl))
            self._details.clear()
        except Exception as e:
            log_error(f"Audit telemetry synchronization failed: {str(e)[:300]}")

# ------------------------------------------------------------------------------
# 4. DATA QUALITY & MERGE UTILITIES
# ------------------------------------------------------------------------------
def add_metadata(df: DataFrame) -> DataFrame:
    return (df
        .withColumn("_row_modified_at", F.from_utc_timestamp(F.current_timestamp(), PipelineConfig.TZ))
        .withColumn("_pipeline_run_id", F.lit(PipelineConfig.RUN_ID)))

def get_merge_metrics(*, spark: SparkSession, table_name: str) -> Dict[str, int]:
    try:
        row = (spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 1").select("operationMetrics").first())
        if row and row[0]:
            m = row[0]
            return {"updated": int(m.get("numTargetRowsUpdated", 0)), "inserted": int(m.get("numTargetRowsInserted", 0))}
    except Exception: pass
    return {"updated": 0, "inserted": 0}

def process_incremental_load(source_schema, source_table, target_schema, target_table, primary_key, watermark_col, filter_condition=None, transform_func=None):
    _start = _est_naive_now()
    load_type = "FULL_OVERWRITE" if RUN_FULL_LOAD else "DELTA_MERGE"
    full_source = f"{PipelineConfig.TARGET_CATALOG}.{source_schema}.{source_table}"
    full_target = f"{PipelineConfig.TARGET_CATALOG}.{target_schema}.{target_table}"
    
    log_info(f"Initiating [{target_schema}] synchronization for entity: [{target_table}]")
    
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PipelineConfig.TARGET_CATALOG}.{target_schema}")
        df_source = spark.table(full_source)
        if filter_condition: df_source = df_source.filter(filter_condition)
        
        if transform_func: df_source = transform_func(df_source)
        
        record_count = df_source.count()
        metrics = {"inserted": 0, "updated": 0}

        if RUN_FULL_LOAD:
            df_source.transform(add_metadata).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_target)
            metrics["inserted"] = record_count
        elif spark.catalog.tableExists(full_target):
            delta_target = DeltaTable.forName(spark, full_target)
            (delta_target.alias("target").merge(
                df_source.transform(add_metadata).alias("source"), f"lower(trim(target.{primary_key})) = lower(trim(source.{primary_key}))"
            ).whenMatchedUpdateAll(condition=f"source.{watermark_col} > target.{watermark_col}").whenNotMatchedInsertAll().execute())
            metrics = get_merge_metrics(spark=spark, table_name=full_target)
        else:
            df_source.transform(add_metadata).write.format("delta").mode("overwrite").saveAsTable(full_target)
            metrics["inserted"] = record_count
            
        log_success(f"[{target_schema}].[{target_table}] SYNC COMPLETE: {metrics['inserted']} inserted.")
        audit.log_detail(entity_name=target_table, layer=target_schema, operation=load_type, row_count=record_count, rows_inserted=metrics["inserted"], rows_updated=metrics["updated"], start_time=_start)
            
    except Exception as e:
        log_error(f"[{target_schema}].[{target_table}] FAILED: {str(e)[:500]}")
        audit.log_detail(entity_name=target_table, layer=target_schema, operation=load_type, row_count=0, status="ERROR", message=str(e)[:2000], start_time=_start)

# ------------------------------------------------------------------------------
# 5. CORE TRANSFORMATION ENGINE (SILVER LAYER)
# ------------------------------------------------------------------------------
def transform_fact_time_and_cost_enriched(df_synerion):
    """
    [Fact Transformation] Synerion & UKG Bridge (Silver Layer)
    
    UPDATES:
    - Renamed 'country_name' to 'employee_country' for better dimensional clarity.
    - Integrated 'auxiliar_codes' lookup for subtask_category (Shrinkage).
    - Maintains L1, L2, L3 organizational hierarchy and HR cost enrichment.
    """
    from pyspark.sql.functions import col, lit, to_date, coalesce, lead, round as spark_round, lower, trim, when, split
    from pyspark.sql.window import Window
    
    # --- 1. Internal Helper: Date Standardization ---
    def safe_date(c):
        return coalesce(col(c).cast("date"), to_date(col(c), "yyyy-MM-dd HH:mm:ss"), to_date(col(c), "M/d/yyyy"))

    df_syn_clean = df_synerion.withColumn("transaction_date", safe_date("transactiondate"))
    
    # --- 2. Master Data Ingestion ---
    df_bridge = spark.read.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_BRONZE_HR}.ukg_employment_details")
    df_history = spark.read.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_BRONZE_HR}.ukg_employee_job_history")
    df_pay_cat = spark.read.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_BRONZE_SYN}.pay_categories")
    df_emp_type = spark.read.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_BRONZE_SYN}.employment_type")
    df_dim_lookup = spark.read.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_BRONZE_HR}.ukg_dimensions_details")
    df_aux_codes = spark.read.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_BRONZE_SYN}.auxiliar_codes")

    # --- 3. Dimensional Normalization: Employee & Country ---
    df_country_lkp = spark.read.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_BRONZE_HR}.ukg_company_details") \
        .select(
            lower(trim(col("companycode"))).alias("c_code"),
            when(col("addresscountry") == "CRI", "Costa Rica")
            .when(col("addresscountry") == "COL", "Colombia")
            .when(col("addresscountry") == "USA", "United States")
            .otherwise(col("addresscountry")).alias("country_resovled") # Nombre temporal
        )

    df_bridge_enriched = df_bridge.alias("emp") \
        .join(df_country_lkp.alias("cnt"), lower(trim(col("emp.companycode"))) == col("cnt.c_code"), "left") \
        .join(df_emp_type.alias("et"), lower(trim(col("emp.employeetypecode"))) == lower(trim(col("et.employee_type_code"))), "left") \
        .select(
            col("emp.employeenumber").cast("string").alias("hr_employee_number"), 
            col("emp.employeeid").alias("ukg_id"),
            col("emp.p_config_name").alias("employee_full_name"),
            col("cnt.country_resovled").alias("employee_country"), # <--- RENOMBRADO AQUÍ
            col("et.reporting_category").alias("employee_reporting_category") 
        )

    # --- 4. Transaction Enrichment & SCD2 Mapping ---
    df_hist_scd2 = df_history.withColumn("effective_date", safe_date("jobEffectiveDate")) \
        .withColumn("end_date", coalesce(lead("effective_date").over(Window.partitionBy("employeeId").orderBy("effective_date")), to_date(lit("2099-12-31"), "yyyy-MM-dd")))

    df_pay_cat_enriched = df_pay_cat.select(
        col("category_number"),
        coalesce(col("reporting_description"), col("description"), lit("Uncategorized")).alias("pay_reporting_desc") 
    )

    # --- 5. Integration ---
    df_enriched = df_syn_clean.alias("syn") \
        .join(df_bridge_enriched.alias("brg"), lower(trim(col("syn.employeeid").cast("string"))) == lower(trim(col("brg.hr_employee_number"))), "left") \
        .join(df_pay_cat_enriched.alias("cat"), col("syn.paycategoryid") == col("cat.category_number"), "left") \
        .join(df_aux_codes.alias("aux"), lower(trim(col("syn.level4_task"))) == lower(trim(col("aux.auxiliar_id"))), "left") \
        .join(df_hist_scd2.alias("hist"), (col("brg.ukg_id") == col("hist.employeeId")) & (col("syn.transaction_date") >= col("hist.effective_date")) & (col("syn.transaction_date") < col("hist.end_date")), "left")

    for i in range(1, 5):
        dim_lvl = df_dim_lookup.filter(col("level") == i).select(col("code").alias(f"c{i}"), col("description").alias(f"d{i}"))
        df_enriched = df_enriched.join(dim_lvl, lower(trim(col(f"hist.orgLevel{i}Code"))) == lower(trim(col(f"c{i}"))), "left") \
                           .withColumnRenamed(f"d{i}", f"org_level_{i}_desc")

    # --- 6. Final Projection ---
    return df_enriched.withColumn(
        "L1", split(col("org_level_1_desc"), "-").getItem(0)
    ).withColumn(
        "L2", split(col("org_level_1_desc"), "-").getItem(1)
    ).withColumn(
        "L3", split(col("org_level_1_desc"), "-").getItem(2)
    ).select(
        col("syn.transaction_date"),
        col("syn.employeeid").alias("synerion_emp_id"),
        col("brg.ukg_id").alias("ukg_employee_id"),
        col("brg.employee_full_name"),
        col("brg.employee_country"), # <--- YA SALE COMO EMPLOYEE_COUNTRY DESDE AQUÍ
        col("brg.employee_reporting_category"),
        col("syn.projectid"),
        col("syn.projecttaskid"),
        col("syn.bookableresourceid"),
        col("syn.paycategoryid").alias("pay_category_id"),
        col("cat.pay_reporting_desc").alias("pay_category_description"),
        
        when(col("aux.synerion_auxiliar_name") == "Default", "Regular Time")
        .otherwise(coalesce(col("aux.synerion_auxiliar_name"), lit("Productive Time")))
        .alias("subtask_category"),
        
        col("syn.paycategorydecimal").alias("worked_hours"),
        spark_round(col("syn.paycategorydecimal") * coalesce(col("hist.hourlyPayRate"), lit(0.0)), 2).alias("actual_cost_usd"),
        col("L1"), col("L2"), col("L3"),
        coalesce(col("org_level_2_desc"), col("hist.orgLevel2Code")).alias("customer"),
        coalesce(col("org_level_3_desc"), col("hist.orgLevel3Code")).alias("service_line"),
        coalesce(col("org_level_4_desc"), col("hist.orgLevel4Code")).alias("industry")
    )

# ------------------------------------------------------------------------------
# 6. METADATA: DATASET REFRESH TRACKING
# ------------------------------------------------------------------------------
def update_dataset_refresh_timestamp():
    """
    Updates the unique metadata table for tracking the Synerion-UKG ETL cycle.
    Used by the Power BI Semantic Model for 'Data as of' visuals.
    """
    _start = _est_naive_now()
    try:
        log_info(f"Generating unique ETL completion timestamp for Synerion-UKG...")
        df_refresh = spark.sql(f"""
            SELECT 
                from_utc_timestamp(current_timestamp(), '{PipelineConfig.TZ}') as LastRefresh,
                'Synerion-UKG Pipeline Complete' as Status,
                '{PipelineConfig.PIPELINE_NAME}' as PipelineSource,
                current_timestamp() as System_UTC_Time
        """)
        target_table = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_SILVER}.info_synerion_ukg_refresh"
        df_refresh.transform(add_metadata).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
        log_success(f"Timestamp updated in: '{target_table}'.")
        
        audit.log_detail(
            entity_name="info_synerion_ukg_refresh", layer=PipelineConfig.SCHEMA_SILVER, 
            operation="FULL_OVERWRITE", row_count=1, status="OK", start_time=_start
        )
    except Exception as e:
        log_error(f"Failed to update refresh timestamp: {str(e)[:300]}")

# ------------------------------------------------------------------------------
# 7. PIPELINE ORCHESTRATION & MAINTENANCE
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    audit = AuditManager(spark=spark, config=PipelineConfig)
    audit.log_run_start()
    log_info(f"Pipeline {PipelineConfig.PIPELINE_NAME} initiated | RunID: {PipelineConfig.RUN_ID}")

    try:
        # 1. Silver Layer Synchronization: Synerion + UKG Enrichment
        process_incremental_load(
            source_schema=PipelineConfig.SCHEMA_BRONZE_SYN,
            source_table="time_costing_detail",
            target_schema=PipelineConfig.SCHEMA_SILVER,
            target_table="fact_time_and_cost_enriched",
            primary_key="synerion_emp_id",
            watermark_col="record_timestamp",
            transform_func=transform_fact_time_and_cost_enriched
        )

        # 2. Dataset Refresh Timestamp Metadata
        update_dataset_refresh_timestamp()

        # 3. Delta Lake Maintenance & Audit Finalization
        audit.log_run_end(status="SUCCESS")
        spark.sql(f"OPTIMIZE {PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_AUDIT}.run_audit_detail")
        spark.sql(f"OPTIMIZE {PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_AUDIT}.run_audit_log")
        log_success(f"Pipeline {PipelineConfig.PIPELINE_NAME} completed successfully.")

    except Exception as ex:
        _err = traceback.format_exc()
        try: audit.log_run_end(status="FAILURE", error_message=_err)
        except Exception: log_error("CRITICAL: Failed to write audit finalization.")
        raise

StatementMeta(, 6c6a0ee4-6592-44b0-bec6-3c8e9f3a3ef9, 3, Finished, Available, Finished, False)

2026-11-03 11:38:59 - [INFO] - Pipeline NB_Corp_Finance_DataHub_ELT_Synerion_UKG initiated | RunID: 3a096ae6-da23-4aed-b023-2519402715e5
2026-11-03 11:38:59 - [INFO] - Initiating [silver] synchronization for entity: [fact_time_and_cost_enriched]
2026-11-03 11:39:37 - [SUCCESS] - [silver].[fact_time_and_cost_enriched] SYNC COMPLETE: 1319654 inserted.
2026-11-03 11:39:39 - [INFO] - Generating unique ETL completion timestamp for Synerion-UKG...
2026-11-03 11:39:42 - [SUCCESS] - Timestamp updated in: 'LH_Corp_DataHub.silver.info_synerion_ukg_refresh'.
2026-11-03 11:40:00 - [SUCCESS] - Pipeline NB_Corp_Finance_DataHub_ELT_Synerion_UKG completed successfully.
